In [1]:
# Essential imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_selection import SelectKBest, f_regression, SelectFromModel, RFE
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

# Set a random state for reproducibility
RANDOM_STATE = 710

In [2]:
# set_config(transform_output="pandas") tells scikit-learn to return transformed features as a pandas DataFrame instead of a NumPy array.
# That is useful because the result keeps readable column names after preprocessing.
from sklearn import set_config
set_config(transform_output='pandas')

In [3]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, root_mean_squared_error, r2_score
# Import data

url = "https://docs.google.com/spreadsheets/d/1IbAAe8x9inoiLXg88fDjKV0xTN1U6onrCPHuvseNQI4/edit?gid=0#gid=0$0"

sheet_id = url.split("/d/")[1].split("/")[0]
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

# naming our dataset housing_regression_data
housing_regression_data = pd.read_csv(csv_url)

# X and y creation
X = housing_regression_data.drop(columns="Id", errors="ignore")
y = X.pop("SalePrice")



# data splitting with 20% as test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_train_log = np.log1p(y_train)

In [17]:
## Pre processing, scaler and Pipelines

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# MSSubClass contains numeric codes, not a genuine numeric measurement.
# Convert it to text so it is treated as a category.
X_train["MSSubClass"] = X_train["MSSubClass"].astype(str)


# Define features from training data only
num_feat = X_train.select_dtypes(include="number").columns
cat_feat = X_train.select_dtypes(exclude="number").columns

# Define ordinal categorical columns
ordinal_categories = {
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtExposure": ["NA", "No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["NA", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["NA", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "FireplaceQu": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageFinish": ["NA", "Unf", "RFn", "Fin"],
    "GarageQual": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "PoolQC": ["NA", "Fa", "TA", "Gd", "Ex"],
}

# Names of columns that will receive ordinal encoding.
ord_feat = list(ordinal_categories)

# Retain only ordinal columns that actually exist in your training data.
ord_feat = [col for col in ord_feat if col in X_train.columns]

# All remaining categorical columns are nominal: their categories
# do not have a meaningful numeric order, so they use one-hot encoding.
oh_feat = [col for col in cat_feat if col not in ord_feat]

# The imputer emits "N_A", so include it in every encoder category list.
ord_categories = [
    ["N_A", *ordinal_categories[col]]
    for col in ord_feat
]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="N_A")),
    ("encoder", OrdinalEncoder(
        categories=ord_categories,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

nominal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",

        # Groups extremely rare categories together.
        # This can reduce overfitting in small training datasets.
        min_frequency=10,
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, num_feat),
    ("ordinal", ordinal_pipe, ord_feat),
    ("nominal", nominal_pipe, oh_feat),
])






In [5]:
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),

    # A forest is a collection of decision trees.
    ("model", RandomForestRegressor(
        random_state=710,
        n_jobs=-1  # use available CPU cores while fitting the forest
    ))
])
rf_pipe


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('ordinal', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [6]:
# Create a simple pipeline for the baseline model


# Evaluate using 5-fold cross-validation
baseline_scores = cross_val_score(
    rf_pipe,
    X_train,
    y_train_log,
    cv=5,
    scoring='neg_root_mean_squared_error'
)
positive_baseline_scores = -baseline_scores

print(f"Baseline Mean CV RMSE: {positive_baseline_scores.mean():.3f}")

Baseline Mean CV RMSE: 0.144


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor
)
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

base_models = [
    ("rf", RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=2,
        random_state=710,
        n_jobs=1
    )),
    ("gbr", GradientBoostingRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=3,
        min_samples_leaf=5,
        random_state=710
    )),
    ("svr", SVR(
        kernel="linear",
        C=1.0
    )),
]

stacked = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression(),
    cv=5,
    n_jobs=-1
)

stacking_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", stacked)
])




In [8]:
stacking_pipe.fit(X_train, y_train_log)

pred_log = stacking_pipe.predict(X_test)
predictions = np.expm1(pred_log)



/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


In [9]:
stack_cv = -cross_val_score(
    stacking_pipe,
    X_train,
    y_train_log,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=1
)

print(f"Stacking CV log-RMSE: {stack_cv.mean():.4f}")
print(f"Standard deviation: {stack_cv.std():.4f}")

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


Stacking CV log-RMSE: 0.1311
Standard deviation: 0.0208


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


In [10]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, root_mean_squared_error, r2_score
# Import data

url = "https://docs.google.com/spreadsheets/d/1tT5Dl8UkyWvuzMHPIzxKg-EXdojOlV_u54Ir4Os8xmo/edit?gid=0#gid=0$0"

sheet_id = url.split("/d/")[1].split("/")[0]
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

# naming our dataset housing_regression_data
test_housing_regression_data = pd.read_csv(csv_url)



In [11]:
id_column = test_housing_regression_data.pop('Id')

In [12]:
pred_log = stacking_pipe.predict(test_housing_regression_data)
predictions = np.expm1(pred_log)

results = pd.DataFrame({
    'Id':id_column,
    'SalePrice':predictions
})

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


In [27]:
results.to_csv('submission_rf_1830.csv',index=False)